# Assignment 02: Building ResNet-34 and ResNet-50 from Scratch
### Course: Computer Vision | Framework: TensorFlow / Keras (Functional API)
---
## Student Information
**Please fill in your details:**
- **Name:** Zimo Guo
- **Student ID:** 20233006327 
- **Partner's Name (if you work in pair):** [Partner's Name]
- **Partner's Student ID (if applicable):** [Partner's ID]

---

## Lab Information
- **Due Date:** April 08, 2026
- **Weight:** 5% of total course grade
- **Platform:** Google Colab, Kaggle, or VSCode (with Jupyter extension)

---

## What You Will Build

In this assignment, you will implement **ResNet-34** and **ResNet-50** step by step.

ResNet uses **skip connections** (residual connections) to allow gradients to flow easily during training. This solves the **vanishing gradient problem** in very deep networks.

You already saw **ResNet-18** in class. It uses two types of blocks:
- `identity_block` — used when input and output shapes are **the same**
- `projection_block` — used when shapes **change** (different channels or smaller spatial size)

In this assignment you will:
- Re-implement the same two blocks and use them to build **ResNet-34**
- Add two new bottleneck blocks — `bottleneck_identity()` and `bottleneck_projection()` — and use them to build **ResNet-50**

---

## Tasks

| Task | What you build |
|------|----------------|
| Task 0 | Warm-up questions (no coding) |
| Task 1 | `identity_block()` |
| Task 2 | `projection_block()` |
| Task 3 | `build_resnet34()` |
| Task 4 | `bottleneck_identity()` and `bottleneck_projection()` |
| Task 5 | `build_resnet50()` + compare with official Keras model |

---

## Rules

- Do **NOT** use `tf.keras.applications.ResNet34` or `ResNet50` to build the model.
- You **may** use the official model **only** in Task 5 for comparison.
- You must be able to explain every line of your code.

---

## Test Image

Download and save this image as **`dog.jpg`** in the same folder as this notebook:

[German Shepherd image](https://upload.wikimedia.org/wikipedia/commons/d/d0/German_Shepherd_-_DSC_0346_%2810096362833%29.jpg)

You will use it in Task 5 to test your ResNet-50.

---
## ️ Setup — Run This Cell First

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("TensorFlow version:", tf.__version__)
print("Setup complete ")

---
## Task 0 — Warm-Up: Understand the Blocks (No Coding)

Before writing code, study the diagrams and answer the questions below.
Use your lecture slides and notes. You can add images here by yourself (Create an image folder and load them here).

---

### Architecture Diagrams

<!-- **Identity Block:**

 *Add your diagram here — Example:* ![identity_block](./images/identity_block.png)

**Projection Block:**

 *Add your diagram here — Example:* ![projection_block](images/projection_block.png)

**Bottleneck Block (ResNet-50):**

 *Add your diagram here — Example:* ![bottleneck](images/bottleneck_block.png)

**Full ResNet-34 Architecture:**

 *Add your diagram here — Example:* ![resnet34](images/resnet34.png)

**Full ResNet-50 Architecture:**

 *Add your diagram here — Example:* ![resnet50](images/resnet50.png) -->

---

### Questions — Edit this cell to write your answers

**Q1.** An `identity_block` receives input shape `(56, 56, 64)` with `filters=64`. What is the output shape?

> **Answer:** 

**Q2.** A `projection_block` receives `(56, 56, 64)` with `filters=128` and `strides=2`. What is the output shape?

> **Answer:** ___

**Q3.** Why does `projection_block` need a 1×1 Conv on the skip connection, but `identity_block` does not?

> **Answer:** ___

**Q4.** A `bottleneck_identity` or `bottleneck_projection` block has `filters=64`. What is the output channel count? (Hint: expansion = 4)

> **Answer:** ___

**Q5.** ResNet-18 has `[2, 2, 2, 2]` blocks per stage. ResNet-34 has `[3, 4, 6, 3]`. How many total residual blocks does ResNet-34 have?

> **Answer:** ___

---
## Reference: How ResNet-18 Was Built in Class

Read this carefully. Your code for ResNet-34 will follow the **exact same pattern**.

In ResNet-18, each stage follows this rule:
- **First block of a new stage** → `projection_block` (shapes change)
- **All other blocks in the same stage** → `identity_block` (shapes stay the same)
- **Exception: Stage 1** has no shape change, so all blocks are `identity_block`

```
ResNet-18 block pattern:
  Stage 1: identity → identity                          (64 filters,  stride=1)  [2 blocks]
  Stage 2: projection → identity                        (128 filters, stride=2)  [2 blocks]
  Stage 3: projection → identity                        (256 filters, stride=2)  [2 blocks]
  Stage 4: projection → identity                        (512 filters, stride=2)  [2 blocks]

ResNet-34 block pattern (same idea, just more blocks):
  Stage 1: identity → identity → identity               (64 filters,  stride=1)  [3 blocks]
  Stage 2: projection → identity → identity → identity  (128 filters, stride=2)  [4 blocks]
  Stage 3: projection → identity × 5                    (256 filters, stride=2)  [6 blocks]
  Stage 4: projection → identity → identity             (512 filters, stride=2)  [3 blocks]
```

---
## Task 1 — Implement `identity_block()`

Used when **input and output shapes are the same**.
The skip connection is a **direct wire** — no transformation needed.

```
Input (same dimensions)
  |
  |---> Conv → BatchNorm → ReLU → Conv → BatchNorm → (+) → ReLU
  |                                                     |
  |_____________________________________________________|
                    (identity shortcut)
```

### Complete the function below

In [ ]:
def identity_block(x, filters, name=''):
    """
    Identity block: input and output shapes are the SAME.
    The skip connection is a direct wire — no Conv needed.

    Args:
        x       : input tensor
        filters : number of filters for both Conv layers
        name    : prefix for layer names (helps with model.summary())

    Returns:
        Output tensor — same shape as input
    """

    # Save input for the skip connection
    shortcut = x

    # ---- Main Path ----

    # First Conv: 3×3, stride=1
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name=name + '_conv1')(x)
    x = layers.BatchNormalization(name=name + '_bn1')(x)
    x = layers.Activation('relu', name=name + '_relu1')(x)

    # TODO: Second Conv — same structure as above
    # Important: NO ReLU here — it comes after the Add below
    # YOUR CODE HERE
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name=name + '_conv2')(x)
    x = layers.BatchNormalization(name=name + '_bn2')(x)

    # ---- Skip Connection ----
    # Shapes already match — just add directly
    x = layers.Add(name=name + '_add')([x, shortcut])
    x = layers.Activation('relu', name=name + '_relu_out')(x)

    return x

### Test Task 1

In [ ]:
# Build a small test model
inp = layers.Input(shape=(56, 56, 64))
out = identity_block(inp, filters=64, name='test_id')
test_m = models.Model(inp, out)

result = test_m(tf.zeros([1, 56, 56, 64]))
print("identity_block output shape:", result.shape)  # Expected: (1, 56, 56, 64)

if result.shape == (1, 56, 56, 64):
    print("Task 1 PASSED!")
else:
    print("Task 1 FAILED — check your second Conv layer.")

---
## Task 2 — Implement `projection_block()`

Used when **shapes change** — the spatial size shrinks (stride=2) and/or channels increase.

Because the skip connection and the main path now have **different shapes**, we must apply a **1×1 Conv + BN** to the shortcut to make the shapes match before adding.

```
Input (different dimensions)
  |
  |---> Conv(stride=2) → BN → ReLU → Conv → BN → (+) → ReLU
  |                                            |
  |---> Conv(1×1, stride=2) → BN --------------|
           (projection shortcut)
```

### Complete the function below

In [ ]:
def projection_block(x, filters, strides=2, name=''):
    """
    Projection block: input and output shapes are DIFFERENT.
    A 1×1 Conv is applied to the skip connection to match shapes.

    Args:
        x       : input tensor
        filters : number of filters for the Conv layers
        strides : stride for the first Conv (default=2 to downsample)
        name    : prefix for layer names

    Returns:
        Output tensor with shape (H/strides, W/strides, filters)
    """

    # ---- Skip Connection (Projection) ----
    # 1×1 Conv to match the output shape of the main path
    shortcut = layers.Conv2D(filters, kernel_size=1, strides=strides, padding='same',
                             use_bias=False, kernel_initializer='he_normal',
                             name=name + '_proj_conv')(x)
    shortcut = layers.BatchNormalization(name=name + '_proj_bn')(shortcut)
    

    # ---- Main Path ----

    # TODO: First Conv — 3×3, use the given strides, then BN, then ReLU
    # YOUR CODE HERE
    x = layers.Conv2D(filters, kernel_size=1, strides=2, padding='same',
                             use_bias=False, kernel_initializer='he_normal',
                             name=name + '_proj_conv')(x)
    x = layers.BatchNormalization(name=name + '_proj_bn')(shortcut)
    x = layers.Activation('relu', name=name + '_relu')(shortcut)

    # TODO: Second Conv — 3×3, stride=1 (no more downsampling), then BN
    # No ReLU here — it comes after the Add below
    # YOUR CODE HERE
    x = layers.Conv2D(filters, kernel_size=1, strides=1, padding='same',
                             use_bias=False, kernel_initializer='he_normal',
                             name=name + '_proj_conv')(x)
    x = layers.BatchNormalization(name=name + '_proj_bn')(shortcut)

    # ---- Add + ReLU ----
    x = layers.Add(name=name + '_add')([x, shortcut])
    x = layers.Activation('relu', name=name + '_relu_out')(x)

    return x

### Test Task 2

In [ ]:
inp = layers.Input(shape=(56, 56, 64))
out = projection_block(inp, filters=128, strides=2, name='test_proj')
test_m2 = models.Model(inp, out)

result = test_m2(tf.zeros([1, 56, 56, 64]))
print("projection_block output shape:", result.shape)  # Expected: (1, 28, 28, 128)

if result.shape == (1, 28, 28, 128):
    print(" Task 2 PASSED!")
else:
    print("Task 2 FAILED — check your Conv layers or shortcut.")

---
## Task 3 — Build `build_resnet34()`

Assemble the full **ResNet-34** using your two blocks.

### ResNet-34 Architecture Table

| Layer / Stage | Blocks | Filters | Stride | Block Pattern | Output Shape |
|---------------|--------|---------|--------|---------------|--------------|
| conv1 | — | 64 | 2 | Conv 7×7 + BN + ReLU + MaxPool | (56, 56, 64) |
| Stage 1 | 3 | 64 | 1 | identity × 3 | (56, 56, 64) |
| Stage 2 | 4 | 128 | 2 | projection + identity × 3 | (28, 28, 128) |
| Stage 3 | 6 | 256 | 2 | projection + identity × 5 | (14, 14, 256) |
| Stage 4 | 3 | 512 | 2 | projection + identity × 2 | (7, 7, 512) |
| Output | — | — | — | GlobalAvgPool + Dense(1000) | (1000,) |

### Complete the function below

In [ ]:
def build_resnet34(input_shape=(224, 224, 3), num_classes=1000):
    """
    Build ResNet-34 using the Functional API.
    Uses identity_block and projection_block.
    Block config: [3, 4, 6, 3]

    Args:
        input_shape : (H, W, C) — default ImageNet size (224, 224, 3)
        num_classes : number of output classes — default 1000

    Returns:
        Keras Model
    """

    inputs = layers.Input(shape=input_shape, name='input')

    # ==================== INITIAL LAYER (conv1) ====================
    # Conv 7×7, 64 filters, stride=2  →  (112, 112, 64)
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    # MaxPool 3×3, stride=2  →  (56, 56, 64)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding='same', name='maxpool')(x)

    print("After conv1 + maxpool : (None, 56, 56,  64)")

    # ==================== STAGE 1: 3 blocks, filters=64, stride=1 ====================
    # No downsampling in Stage 1 — all identity blocks
    x = identity_block(x, filters=64, name='s1_b1')
    x = identity_block(x, filters=64, name='s1_b2')
    x = identity_block(x, filters=64, name='s1_b3')

    print("After Stage 1         : (None, 56, 56,  64)")

    # ==================== STAGE 2: 4 blocks, filters=128, stride=2 ====================
    # First block: projection (stride=2 → downsamples and increases channels to 128)
    # Remaining 3 blocks: identity
    x = projection_block(x, filters=128, strides=2, name='s2_b1')

    # TODO: Add 3 identity blocks (filters=128)
    # YOUR CODE HERE
    

    print("After Stage 2         : (None, 28, 28, 128)")

    # ==================== STAGE 3: 6 blocks, filters=256, stride=2 ====================
    # First block: projection, then 5 identity blocks

    # TODO: projection block (filters=256, strides=2)
    # YOUR CODE HERE

    # TODO: 5 identity blocks (filters=256)
    # YOUR CODE HERE

    print("After Stage 3         : (None, 14, 14, 256)")

    # ==================== STAGE 4: 3 blocks, filters=512, stride=2 ====================
    # First block: projection, then 2 identity blocks

    # TODO: projection block (filters=512, strides=2)
    # YOUR CODE HERE

    # TODO: 2 identity blocks (filters=512)
    # YOUR CODE HERE

    print("After Stage 4         : (None,  7,  7, 512)")

    # ==================== OUTPUT ====================
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    print("After GlobalAvgPool   : (None, 512)")

    outputs = layers.Dense(num_classes, activation='softmax',
                           kernel_initializer='he_normal', name='fc')(x)
    print("After fc              : (None,", num_classes, ")")

    model = models.Model(inputs=inputs, outputs=outputs, name='ResNet34')
    return model

### Test Task 3

In [ ]:
resnet34 = build_resnet34(input_shape=(224, 224, 3), num_classes=1000)

print("\n" + "="*50)
total = resnet34.count_params()
out   = resnet34(tf.zeros([1, 224, 224, 3]), training=False)
print(f"Output shape    : {out.shape}")       # Expected: (1, 1000)
print(f"Total parameters: {total:,}")         # Expected: ~21.8 million
print("="*50)

if out.shape == (1, 1000) and total > 20_000_000:
    print("Task 3 PASSED!")
else:
    print("Task 3 FAILED — check your stages.")

In [ ]:
# Optional: print full model summary to see every layer
resnet34.summary()

---
## Task 4 — Implement `bottleneck_identity()` and `bottleneck_projection()`

The bottleneck block is the building unit of **ResNet-50, ResNet-101, ResNet-152**.

Instead of two 3×3 Convs (like ResNet-34), it uses three convolutions: **1×1 → 3×3 → 1×1**

- First **1×1** → reduce channels (less computation)
- **3×3** → main processing
- Last **1×1** → expand channels by ×4 (`expansion = 4`)

---

### Important — Same Rule as ResNet-34 Applies Here Too!

Just like in ResNet-34, you need **two types** of bottleneck blocks:

| Block | When to use | Skip connection |
|---|---|---|
| `bottleneck_identity` | shapes stay the SAME | direct wire (no Conv) |
| `bottleneck_projection` | shapes CHANGE (stride or channels) | 1×1 Conv + BN |

The rule is identical to ResNet-34:
- **First block of each stage** → `bottleneck_projection` (shapes change)
- **All other blocks in the same stage** → `bottleneck_identity` (shapes stay the same)
- **Exception: Stage 1** — shapes still change (64 → 256 channels), so `bottleneck_projection` is used, but stride=1

```
ResNet-50 block pattern:
  Stage 1: bottleneck_projection(stride=1) + bottleneck_identity × 2   [3 blocks, 64→256 ch]
  Stage 2: bottleneck_projection(stride=2) + bottleneck_identity × 3   [4 blocks, 256→512 ch]
  Stage 3: bottleneck_projection(stride=2) + bottleneck_identity × 5   [6 blocks, 512→1024 ch]
  Stage 4: bottleneck_projection(stride=2) + bottleneck_identity × 2   [3 blocks, 1024→2048 ch]
```

---

### `bottleneck_identity` — direct wire shortcut

```
Input (same dimensions throughout)
  |
  |---> Conv(1×1) → BN → ReLU → Conv(3×3) → BN → ReLU → Conv(1×1) → BN → (+) → ReLU
  |                                                                       |
  |----------------------------------------------------------------- direct wire
```

### `bottleneck_projection` — projected shortcut

```
Input (different dimensions)
  |
  |---> Conv(1×1) → BN → ReLU → Conv(3×3, stride=2) → BN → ReLU → Conv(1×1) → BN → (+) → ReLU
  |                                                                                 |
  |---> Conv(1×1, stride=2) → BN ---------------------------------------------------|
              (projection shortcut)
```

###  Complete both functions below

In [ ]:
def bottleneck_identity(x, filters, name=''):
    """
    Bottleneck block where input and output shapes are the SAME.
    The skip connection is a direct wire — no Conv needed.
    Input channels must already be filters*4.

    Args:
        x       : input tensor  (channels must equal filters*4)
        filters : bottleneck width (output channels = filters * 4)
        name    : prefix for layer names

    Returns:
        Output tensor — same shape as input
    """

    # Save input — direct wire, no transformation needed
    shortcut = x

    # ---- Main Path ----

    # First: 1×1 Conv — REDUCE channels to 'filters'
    x = layers.Conv2D(filters, kernel_size=1, strides=1, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name=name + '_conv1')(x)
    x = layers.BatchNormalization(name=name + '_bn1')(x)
    x = layers.Activation('relu', name=name + '_relu1')(x)

    # TODO: Second — 3×3 Conv, stride=1, then BN, then ReLU
    # YOUR CODE HERE

    # TODO: Third — 1×1 Conv, EXPAND channels to filters*4, stride=1, then BN
    # No ReLU here — it comes after the Add
    # YOUR CODE HERE

    # ---- Skip Connection ----
    # Shapes already match — add directly
    x = layers.Add(name=name + '_add')([x, shortcut])
    x = layers.Activation('relu', name=name + '_relu_out')(x)

    return x


def bottleneck_projection(x, filters, strides=2, name=''):
    """
    Bottleneck block where shapes CHANGE.
    A 1×1 Conv projection is applied to the skip connection.
    Used as the FIRST block of each stage.

    Args:
        x       : input tensor
        filters : bottleneck width (output channels = filters * 4)
        strides : stride for the 3×3 Conv (default=2 to downsample)
        name    : prefix for layer names

    Returns:
        Output tensor with shape (H/strides, W/strides, filters*4)
    """

    # ---- Skip Connection (Projection) ----
    # Output of main path = filters*4 — shortcut must also become filters*4
    shortcut = layers.Conv2D(filters * 4, kernel_size=1, strides=strides, padding='same',
                             use_bias=False, kernel_initializer='he_normal',
                             name=name + '_proj_conv')(x)
    shortcut = layers.BatchNormalization(name=name + '_proj_bn')(shortcut)

    # ---- Main Path ----

    # First: 1×1 Conv — REDUCE channels to 'filters', stride=1
    x = layers.Conv2D(filters, kernel_size=1, strides=1, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name=name + '_conv1')(x)
    x = layers.BatchNormalization(name=name + '_bn1')(x)
    x = layers.Activation('relu', name=name + '_relu1')(x)

    # TODO: Second — 3×3 Conv, apply the given strides here, then BN, then ReLU
    # YOUR CODE HERE

    # TODO: Third — 1×1 Conv, EXPAND channels to filters*4, stride=1, then BN
    # No ReLU here — it comes after the Add
    # YOUR CODE HERE

    # ---- Add + ReLU ----
    x = layers.Add(name=name + '_add')([x, shortcut])
    x = layers.Activation('relu', name=name + '_relu_out')(x)

    return x

### Test Task 4

In [ ]:
# ── Test bottleneck_identity ────────────────────────────────────────────
# Input channels must already be filters*4 = 256
inp_id = layers.Input(shape=(56, 56, 256))
r_id   = models.Model(inp_id, bottleneck_identity(inp_id, filters=64, name='ti'))(tf.zeros([1,56,56,256]))
print("bottleneck_identity (filters=64, input_ch=256):", r_id.shape)  # Expected: (1, 56, 56, 256)

# ── Test bottleneck_projection — stride=1, channel change ───────────────────
# Stage 1 first block: input=64 ch (from maxpool), output = 64*4 = 256, stride=1
inp_p1 = layers.Input(shape=(56, 56, 64))
r_p1   = models.Model(inp_p1, bottleneck_projection(inp_p1, filters=64, strides=1, name='tp1'))(tf.zeros([1,56,56,64]))
print("bottleneck_projection (filters=64,  stride=1):", r_p1.shape)   # Expected: (1, 56, 56, 256)

# ── Test bottleneck_projection — stride=2, spatial + channel change ──────────
# Stage 2 first block: input=256 ch, output = 128*4 = 512, H/W halved
inp_p2 = layers.Input(shape=(56, 56, 256))
r_p2   = models.Model(inp_p2, bottleneck_projection(inp_p2, filters=128, strides=2, name='tp2'))(tf.zeros([1,56,56,256]))
print("bottleneck_projection (filters=128, stride=2):", r_p2.shape)   # Expected: (1, 28, 28, 512)

if r_id.shape==(1,56,56,256) and r_p1.shape==(1,56,56,256) and r_p2.shape==(1,28,28,512):
    print("\nTask 4 PASSED!")
else:
    print("\nTask 4 FAILED — check conv2 (3×3) or conv3 (1×1 expand).")

---
## Task 5 — Build `build_resnet50()` and Compare with Official Keras Model

Assemble the full **ResNet-50** using your `bottleneck_block`.

### ResNet-50 Architecture Table

*Add your diagram here — Example:* `![ResNet-50](images/resnet50.png)`

| Layer / Stage | Blocks | Filters | Stride | Block Pattern | Output Shape |
|---------------|--------|---------|--------|---------------|--------------|
| conv1 | — | 64 | 2 | Conv 7×7 + BN + ReLU + MaxPool | (56, 56, 64) |
| Stage 1 | 3 | 64 | 1 | bottleneck_projection(s=1) + bottleneck_identity × 2 | (56, 56, **256**) |
| Stage 2 | 4 | 128 | 2 | bottleneck_projection(s=2) + bottleneck_identity × 3 | (28, 28, **512**) |
| Stage 3 | 6 | 256 | 2 | bottleneck_projection(s=2) + bottleneck_identity × 5 | (14, 14, **1024**) |
| Stage 4 | 3 | 512 | 2 | bottleneck_projection(s=2) + bottleneck_identity × 2 | (7, 7, **2048**) |
| Output | — | — | — | GlobalAvgPool + Dense(1000) | (1000,) |

**Compare with ResNet-34 — notice the perfect parallel:**
- Same block config `[3, 4, 6, 3]` 
- Same filters per stage `[64, 128, 256, 512]` 
- Same rule: first block of each stage = projection, rest = identity 
- Only difference: 3-layer bottleneck structure (1×1→3×3→1×1) instead of 2-layer (3×3→3×3) 
- Output channels are 4× larger because of expansion 

### Part A — Complete `build_resnet50()`

In [ ]:
def build_resnet50(input_shape=(224, 224, 3), num_classes=1000):
    """
    Build ResNet-50 using the Functional API.
    Uses bottleneck_block throughout.
    Block config: [3, 4, 6, 3]

    Args:
        input_shape : (H, W, C) — default ImageNet size (224, 224, 3)
        num_classes : number of output classes — default 1000

    Returns:
        Keras Model
    """

    inputs = layers.Input(shape=input_shape, name='input')

    # ==================== INITIAL LAYER (conv1) ====================
    # Same as ResNet-34: Conv 7×7 → BN → ReLU → MaxPool
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding='same', name='maxpool')(x)

    print("After conv1 + maxpool : (None, 56, 56,   64)")

    # ==================== STAGE 1: 3 blocks, filters=64, stride=1 ====================
    # First block: bottleneck_projection — stride=1 but channels change (64 → 256)
    # Remaining 2 blocks: bottleneck_identity — shapes stay the same (256 → 256)
    x = bottleneck_projection(x, filters=64, strides=1, name='s1_b1')
    x = bottleneck_identity  (x, filters=64,            name='s1_b2')
    x = bottleneck_identity  (x, filters=64,            name='s1_b3')

    print("After Stage 1         : (None, 56, 56,  256)")

    # ==================== STAGE 2: 4 blocks, filters=128, stride=2 ====================
    # First block: bottleneck_projection — stride=2 + channels change (256 → 512)
    # Remaining 3 blocks: bottleneck_identity — shapes stay the same (512 → 512)

    # TODO: bottleneck_projection (filters=128, strides=2) — first block
    # YOUR CODE HERE

    # TODO: 3 bottleneck_identity blocks (filters=128)
    # YOUR CODE HERE

    print("After Stage 2         : (None, 28, 28,  512)")

    # ==================== STAGE 3: 6 blocks, filters=256, stride=2 ====================
    # First block: bottleneck_projection — stride=2 + channels change (512 → 1024)
    # Remaining 5 blocks: bottleneck_identity — shapes stay the same (1024 → 1024)

    # TODO: bottleneck_projection (filters=256, strides=2) — first block
    # YOUR CODE HERE

    # TODO: 5 bottleneck_identity blocks (filters=256)
    # YOUR CODE HERE

    print("After Stage 3         : (None, 14, 14, 1024)")

    # ==================== STAGE 4: 3 blocks, filters=512, stride=2 ====================
    # First block: bottleneck_projection — stride=2 + channels change (1024 → 2048)
    # Remaining 2 blocks: bottleneck_identity — shapes stay the same (2048 → 2048)

    # TODO: bottleneck_projection (filters=512, strides=2) — first block
    # YOUR CODE HERE

    # TODO: 2 bottleneck_identity blocks (filters=512)
    # YOUR CODE HERE

    print("After Stage 4         : (None,  7,  7, 2048)")

    # ==================== OUTPUT ====================
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    print("After GlobalAvgPool   : (None, 2048)")

    outputs = layers.Dense(num_classes, activation='softmax',
                           kernel_initializer='he_normal', name='fc')(x)
    print("After fc              : (None,", num_classes, ")")

    model = models.Model(inputs=inputs, outputs=outputs, name='ResNet50')
    return model

### Part B — Build ResNet-50 and Check

In [ ]:
resnet50_custom = build_resnet50(input_shape=(224, 224, 3), num_classes=1000)

print("\n" + "="*50)
total = resnet50_custom.count_params()
out   = resnet50_custom(tf.zeros([1, 224, 224, 3]), training=False)
print(f"Output shape    : {out.shape}")       # Expected: (1, 1000)
print(f"Total parameters: {total:,}")         # Expected: ~25.6 million
print("="*50)

if out.shape == (1, 1000) and total > 23_000_000:
    print("Task 5 Part B PASSED!")
else:
    print("Task 5 Part B FAILED — check your stages.")

### Part C — Load the Dog Image

In [ ]:
# Make sure dog.jpg is in the same folder as this notebook
img = Image.open('dog.jpg').convert('RGB').resize((224, 224))

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Test Image: dog.jpg')
plt.tight_layout()
plt.show()

# Preprocess for your custom model
# Normalize using ImageNet mean and std
img_array = np.array(img, dtype=np.float32) / 255.0
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
img_array  = (img_array - mean) / std
img_tensor = tf.expand_dims(img_array, axis=0)   # shape: (1, 224, 224, 3)

print("Image preprocessed. Shape:", img_tensor.shape)

### Part D — Run Official Keras ResNet-50 (Pretrained)

In [ ]:
from tensorflow.keras.applications import ResNet50 as KerasResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions

official_resnet50 = KerasResNet50(weights='imagenet')
print("Official Keras ResNet-50 loaded") 
print(f"Official parameters: {official_resnet50.count_params():,}")

# Official model uses its own preprocessing — do NOT use the tensor from Part C
img_official = np.array(Image.open('dog.jpg').convert('RGB').resize((224, 224)),
                        dtype=np.float32)
img_official = preprocess_input(np.expand_dims(img_official, axis=0))

preds = official_resnet50.predict(img_official, verbose=0)
top5  = decode_predictions(preds, top=5)[0]

print("\n Official Keras ResNet-50 — Top-5 Predictions:")
print("-" * 50)
for rank, (_, label, prob) in enumerate(top5, 1):
    bar = '|' * int(prob * 40)
    print(f"  {rank}. {label:<28} {prob*100:5.2f}%  {bar}")

### Part E — Run Your Custom ResNet-50

In [ ]:
# Your model has random weights → predictions will be random (this is expected!)
probs_custom = resnet50_custom(img_tensor, training=False).numpy()[0]
top5_idx     = np.argsort(probs_custom)[::-1][:5]

print(" Your Custom ResNet-50 (random weights) — Top-5 Predictions:")
print("-" * 55)
for rank, idx in enumerate(top5_idx, 1):
    bar = '|' * int(probs_custom[idx] * 40)
    print(f"  {rank}. Class index {idx:<6}  {probs_custom[idx]*100:5.2f}%  {bar}")

print("\n  Random predictions are expected.")
print("    Your model is correctly built — it just has not been trained yet.")

### Part F — Final Comparison Table

In [ ]:
p34       = resnet34.count_params()
p50_yours = resnet50_custom.count_params()
p50_offic = official_resnet50.count_params()

print("=" * 55)
print("            Model Parameter Comparison")
print("=" * 55)
print(f"  {'Model':<35} {'Parameters':>12}")
print("-" * 55)
print(f"  {'Your ResNet-34':<35} {p34:>12,}")
print(f"  {'Your ResNet-50 (custom)':<35} {p50_yours:>12,}")
print(f"  {'Official Keras ResNet-50':<35} {p50_offic:>12,}")
print("=" * 55)

diff = abs(p50_yours - p50_offic)
if diff < 500_000:
    print("\n Your ResNet-50 architecture is correct!")
else:
    print(f"\n  Parameter difference: {diff:,} — recheck your stage configuration.")

print("\nNote: ResNet-34 and ResNet-50 both use [3,4,6,3] blocks per stage.")
print("ResNet-50 is larger because bottleneck blocks expand channels by 4×.")

---
##  Final Reflection

Answer the questions below by editing this cell.

**Q1.** You used `identity_block` and `projection_block` for both ResNet-18 (in class) and ResNet-34 (this assignment). What was the only difference between the two models?

> **Answer:** ___

**Q2.** Why does `projection_block` need a **1×1 Conv** on the skip connection, but `identity_block` does not?

> **Answer:** ___

**Q3.** ResNet-34 and ResNet-50 both use `[3, 4, 6, 3]` blocks per stage and the same filter sizes. Why does ResNet-50 have more parameters?

> **Answer:** ___

**Q4.** You implemented `bottleneck_identity` and `bottleneck_projection` separately. Why does `bottleneck_projection` need a 1×1 Conv on the skip connection, but `bottleneck_identity` does not? Give an example of when each is used in ResNet-50.

> **Answer:** ___

**Q5.** Your custom ResNet-50 predicted random classes, but the official model correctly identified the dog. What is missing from your model?

> **Answer:** ___

---
##  Submission Checklist

Before submitting, check all items:

-  All code cells completed and **executed** (do not clear outputs)
-  Task 0 warm-up questions answered
-  All 5 reflection questions answered
-  `resnet34.summary()` output is visible
-  Final parameter comparison table is visible
-  Dog image prediction output is visible
-  Export your notebook as a HTML file and upload on Canvas

**File name format:** `Assignment02.ipynb`